# Phase 2B-4 · Management scenarios and break-even

Highwood Feeders simulated case study. Amounts are nominal CAD. The booked margin covers purchase, allocated feed, yardage and veterinary costs. Price sensitivities reprice observed activity. Operational scenarios change modeled outcomes using explicit assumptions.

Run all cells with the tracked sources and weather snapshot. Edit the scenario JSON to change the assumptions. The named cases in the decision exhibits are from the supplied configuration. Sweep ranges can be edited below. The [findings report](../phase2b/04_MANAGEMENT_SCENARIOS_FINDINGS.md) gives input bounds, accounting equations and interpretation.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib import ticker
from matplotlib.colors import TwoSlopeNorm

from feedlot_analytics.feed_growth import load_analysis
from feedlot_analytics.management_scenarios import ScenarioEngine
from feedlot_analytics.scenario_config import PriceShocks, Scenario, load_scenarios

root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").exists())
build, deliveries = load_analysis(root)
engine = ScenarioEngine(build, deliveries)
config = load_scenarios(root / "phase2b/04_MANAGEMENT_SCENARIOS_CONFIG.json")
results = {scenario.name: engine.evaluate(scenario) for scenario in config}
booked = engine.evaluate(Scenario("Observed baseline"))
summary = pd.concat([result.summarize() for result in results.values()], ignore_index=True).set_index("scenario")
blue, teal, red, grey = "#215991", "#16796f", "#a43f37", "#657180"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "bold",
                     "axes.labelcolor": "#253447", "text.color": "#253447", "figure.dpi": 125})
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
controls = build.reconciliation["controls"]
base = booked.summarize().iloc[0]
print(f"{len(controls)} controls passed · {len(build.tables['weather_daily'])} verified weather dates")
display(base[["head_placed", "head_sold", "head_mortality", "booked_eligible_head", "placement_weight_kg",
              "booked_hcw_kg", "revenue_cad", "purchase_cost_cad", "allocated_feed_cost_cad",
              "yardage_cost_cad", "vet_cost_cad", "total_cost_cad", "margin_cad",
              "booked_mortality_burden_cad", "margin_cad_per_head_placed"]].to_frame("Observed"))

## Scenario comparison

Loss-making lots have modeled margin below CAD 0 at cent precision across all outcomes in each lot. CAD/head uses all placements, including those that die. Operational rows include the separately displayed incremental other costs when specified.

In [ ]:
financial = summary[["classification", "head_sold", "head_mortality", "revenue_cad", "total_cost_cad",
                     "margin_cad", "margin_cad_per_head_placed", "margin_change_cad",
                     "loss_making_lots", "loss_making_lot_share"]].copy()
financial["loss_making_lot_share"] = financial.loss_making_lot_share.map("{:.1%}".format)
display(financial)
assumptions = summary[["assumptions", "limitations"]].copy()
assumptions["assumptions"] = assumptions.assumptions.map(lambda values: json.dumps(values, ensure_ascii=False))
with pd.option_context("display.max_colwidth", None):
    display(assumptions)

## Purchase and carcass headroom by cohort

Purchase changes apply to 5,084,801.2 kg of booked entry liveweight across all 15,000 purchases. Carcass changes apply to 5,655,693.9 kg HCW across 14,760 eligible sales. A fractional carcass shock reprices eligible booked revenue. Grade and sold volume stay fixed in these cases. The cohort panels use entry quarter and origin. Each ratio uses pooled eligible kilograms.

In [ ]:
purchase = results["Higher feeder basis"]
carcass = results["Lower carcass CAD/kg"]
thresholds = ["booked_purchase_cad_per_live_kg", "max_purchase_cad_per_live_kg",
              "purchase_headroom_cad_per_live_kg", "booked_revenue_cad_per_hcw_kg",
              "min_carcass_cad_per_hcw_kg", "carcass_headroom_cad_per_hcw_kg"]
exposure = ["head_placed", "head_hcw_eligible", "placement_weight_kg", "hcw_kg"]
portfolio_be = engine.break_even()
display(portfolio_be[exposure + thresholds + ["feed_fraction_at_break_even",
                                              "carcass_fraction_at_break_even"]].round(6))

for group in [("origin_region",), ("placement_quarter",), ("origin_region", "placement_quarter")]:
    feeder_view = purchase.summarize(by=group).set_index(list(group))
    carcass_view = carcass.summarize(by=group).set_index(list(group))
    display(engine.break_even(by=group)[exposure + thresholds].join(
        feeder_view[["lots", "purchase_price_cost_change_cad", "margin_cad_per_head_placed"]]
        .rename(columns={"margin_cad_per_head_placed": "feeder_shock_cad_per_placed"})).join(
        carcass_view[["carcass_price_revenue_change_cad", "margin_cad_per_head_placed"]]
        .rename(columns={"margin_cad_per_head_placed": "carcass_shock_cad_per_placed"})).round(4))

## One-way price sensitivities

The charts show portfolio modeled margin as one input changes. The lines use observed liveweight, HCW and allocated feed cost. The gray guide locates the booked input. The dot marks zero modeled margin where the change is within the displayed range.

In [ ]:
sweeps = [
    ("purchase_cad_per_live_kg", np.linspace(-.2, 1, 31), 1,
     portfolio_be.iloc[0].purchase_headroom_cad_per_live_kg, "Feeder basis change (CAD/kg live)"),
    ("feed_fraction", np.linspace(-.10, .50, 31), 100,
     portfolio_be.iloc[0].feed_fraction_at_break_even, "Allocated feed cost change (%)"),
    ("carcass_cad_per_hcw_kg", np.linspace(-.95, .25, 31), 1,
     -portfolio_be.iloc[0].carcass_headroom_cad_per_hcw_kg, "Carcass price change (CAD/kg HCW)"),
]
fig, axes = plt.subplots(1, len(sweeps), figsize=(12.0, 3.4), sharey=True, layout="constrained")
for ax, (field, values, unit, threshold, label) in zip(axes, sweeps, strict=True):
    margins = [engine.evaluate(Scenario("One-way", PriceShocks(**{field: value}))).lots.margin_cad.sum()
               for value in values]
    ax.plot(values * unit, np.array(margins) / 1e6, color=blue, linewidth=2.2)
    ax.axvline(0, color=grey, linestyle=":", linewidth=1)
    ax.axhline(0, color=grey, linewidth=.9)
    ax.scatter([threshold * unit], [0], color=red, s=28, zorder=3)
    ax.set_xlabel(label)
    ax.grid(axis="y", alpha=.15)
axes[0].set_ylabel("Modeled margin (CAD million)")
fig.suptitle("One price at a time · 90 closed lots", fontweight="bold")
plt.show()

## Feed costs by phase, ingredient and lot

The engine derives cost shares from lot–pen–day deliveries, scales them to booked lot allocations, and applies the shocks to that allocation. These exhibits show financial exposure with unchanged DM, rations, ADG, FCR and outcomes. The high-feed and low-margin screen uses the upper and lower quartiles of the 90 observed lot budgets per placement, respectively.

In [ ]:
phase_case = results["Finisher feed repriced"]
ingredient_case = results["Concentrate repriced"]
for case, dimension in [(phase_case, "ration_phase"), (ingredient_case, "component")]:
    display(engine.feed_cost_detail(case.scenario.prices).groupby(dimension)[[
        "delivery_attributed_cost_cad", "booked_feed_exposure_cad", "feed_cost_change_cad",
        "scenario_feed_exposure_cad"]].sum().sort_values("booked_feed_exposure_cad", ascending=False))

lot_feed = booked.lots.set_index("lot_id")[["head_placed", "allocated_feed_cost_cad", "margin_cad_per_head_placed"]]
lot_feed["booked_feed_cad_per_placed"] = lot_feed.allocated_feed_cost_cad / lot_feed.head_placed
lot_feed["high_feed"] = lot_feed.booked_feed_cad_per_placed.ge(lot_feed.booked_feed_cad_per_placed.quantile(.75))
lot_feed["low_margin"] = lot_feed.margin_cad_per_head_placed.le(lot_feed.margin_cad_per_head_placed.quantile(.25))
uniform_feed = results["Higher allocated feed"].lots.set_index("lot_id")
lot_feed["feed_shock_cost_cad"] = uniform_feed.feed_price_cost_change_cad
lot_feed["feed_shock_margin_cad_per_placed"] = uniform_feed.margin_cad_per_head_placed
display(lot_feed.groupby(["high_feed", "low_margin"]).agg(
    lots=("head_placed", "size"), placed=("head_placed", "sum"), booked_feed_cad=("allocated_feed_cost_cad", "sum"),
    feed_shock_cost_cad=("feed_shock_cost_cad", "sum"),
))
display(lot_feed[lot_feed.high_feed & lot_feed.low_margin].sort_values("margin_cad_per_head_placed"))

## Two prices at a time and lot downside

The shaded grid shows the modeled portfolio margin for concurrent feeder and carcass price changes at the adverse case feed input. The contours mark break-even at booked feed and at the adverse feed cost. Placement, sale volume, grade mix and mortality stay fixed on this chart. The lot panel starts from the eight lowest booked margins per placement, then shows their exposure to the two adverse cases.

In [ ]:
stress = results["Adverse prices"]
combined = results["Adverse prices and higher mortality"]
stress_prices = stress.scenario.prices
feed_stress = stress_prices.feed_fraction
purchase_axis = np.linspace(-.2, 1.0, 17)
carcass_axis = np.linspace(-1.05, .35, 17)
grid = np.array([
    [engine.evaluate(Scenario("Two-way", PriceShocks(purchase_cad_per_live_kg=purchase_change,
                                                    carcass_cad_per_hcw_kg=carcass_change,
                                                    feed_fraction=feed_stress))).lots.margin_cad.sum() / 1e6
     for purchase_change in purchase_axis]
    for carcass_change in carcass_axis
])
fig, ax = plt.subplots(figsize=(8.9, 4.6), layout="constrained")
norm = TwoSlopeNorm(vcenter=0, vmin=min(-.01, grid.min()), vmax=max(.01, grid.max()))
heat = ax.pcolormesh(purchase_axis, carcass_axis, grid, shading="nearest", cmap="RdBu", norm=norm)
for fraction, linestyle, label in [(0, "--", "Break-even · booked feed"),
                                   (feed_stress, "-", f"Break-even · feed change {feed_stress:+.0%}")]:
    frontier = engine.break_even_frontier(purchase_axis, feed_fraction=fraction)
    ax.plot(frontier.purchase_cad_per_live_kg, frontier.break_even_carcass_cad_per_hcw_kg,
            linestyle=linestyle, color="#172431", linewidth=1.45, label=label)
ax.scatter(0, 0, color="white", edgecolors="#172431", label="Booked price offsets", zorder=4)
ax.scatter(stress_prices.purchase_cad_per_live_kg, stress_prices.carcass_cad_per_hcw_kg,
           marker="D", color="#ffcb69", edgecolors="#172431", label="Adverse price case", zorder=4)
ax.set(xlabel="Feeder basis change (CAD/kg live)", ylabel="Carcass price change (CAD/kg HCW)",
       title=f"Combined price stress · allocated feed change {feed_stress:+.0%}")
ax.legend(loc="upper left", framealpha=.94, fontsize=8)
fig.colorbar(heat, ax=ax, label="Modeled margin (CAD million)")
plt.show()

In [ ]:
lot_base = booked.lots.set_index("lot_id")
risk = lot_base[["origin_region", "placement_quarter", "head_placed", "margin_cad_per_head_placed"]].rename(
    columns={"margin_cad_per_head_placed": "booked_cad_per_placed"})
risk = risk.join(engine.break_even(by=("lot_id",)).reset_index().set_index("lot_id")[
    ["purchase_headroom_cad_per_live_kg", "carcass_headroom_cad_per_hcw_kg"]])
for column, result in [("adverse_cad_per_placed", stress), ("combined_cad_per_placed", combined)]:
    risk[column] = result.lots.set_index("lot_id").margin_cad_per_head_placed
weakest = risk.nsmallest(8, "booked_cad_per_placed")
display(weakest)
display(risk.nsmallest(5, "combined_cad_per_placed"))
show = weakest.iloc[::-1]
fig, ax = plt.subplots(figsize=(8.8, 4), layout="constrained")
ax.axvline(0, linewidth=.9, color=grey)
ax.hlines(show.index, show.combined_cad_per_placed, show.booked_cad_per_placed, color="#bdc8d3", linewidth=1.4)
for column, color, name in [("booked_cad_per_placed", teal, "Booked"), ("adverse_cad_per_placed", blue, "Adverse prices"),
                            ("combined_cad_per_placed", red, "Adverse prices and higher mortality")]:
    ax.scatter(show[column], show.index, color=color, label=name, s=28, zorder=3)
ax.set(title="Eight lots with the lowest booked margin per placement", xlabel="Modeled margin (CAD/head placed)")
ax.legend(loc="upper center", bbox_to_anchor=(.5, -.17), ncols=3, frameon=False, fontsize=8)
ax.grid(axis="x", alpha=.16)
plt.show()

## Mortality accounting bridge

The scenario assigns whole hypothetical outcomes reproducibly across lots from the original sales or deaths. A change in placements is outside these scenarios. A lower death count adds conditional sale proceeds and post-death feeding, yardage, veterinary and other expense. Costs already booked on the selected mortality records stay in the closeout. A higher death count removes the selected sales and credits only the stated avoidable feed and yardage shares. Overlap with market repricing is calculated at scenario prices.

In [ ]:
bridge_rows = []
bridge_amounts = ["booked_death_burden_cad", "hcw_delta_kg", "revenue_delta_cad", "feed_delta_cad",
                  "yardage_delta_cad", "vet_delta_cad", "other_delta_cad", "incremental_margin_cad"]
for name, result in results.items():
    bridge = result.mortality_bridge
    if bridge.empty:
        continue
    bridge_rows.append({"scenario": name, "whole_outcomes_changed": len(bridge),
                        "mean_extra_financial_days": bridge.extra_financial_days.mean(),
                        "mean_conditional_finish_live_kg": bridge.conditional_finish_live_kg.mean(),
                        **bridge[bridge_amounts].sum().to_dict()})
display(pd.DataFrame(bridge_rows).set_index("scenario"))

bridge = results["Lower mortality, conservative finish"].mortality_bridge
values = bridge[["revenue_delta_cad", "feed_delta_cad", "yardage_delta_cad", "vet_delta_cad", "other_delta_cad"]].sum()
amounts = np.r_[values.iloc[0], -values.iloc[1:].to_numpy()] / 1e3
ends = amounts.cumsum()
starts = np.r_[0, ends[:-1]]
positions = np.arange(len(amounts))
labels = ["Sale revenue", "Extra feed", "Extra yardage", "Extra veterinary", "Other", "Net increment"]
fig, ax = plt.subplots(figsize=(9.6, 3.9), layout="constrained")
ax.bar(positions, np.abs(amounts), bottom=np.minimum(starts, ends), color=[teal, red, red, red, red], width=.64)
ax.bar(len(amounts), ends[-1], color=blue, width=.64)
for index, value in enumerate(amounts):
    ax.text(index, max(ends[index], starts[index]) + 3, f"{value:+.1f}", ha="center", fontsize=9)
ax.text(len(amounts), ends[-1] + 3, f"{ends[-1]:+.1f}", ha="center", fontsize=9)
ax.set_ylim(min(0, ends.min()) - 4, max(0, ends.max()) + 13)
ax.axhline(0, color=grey, linewidth=.8)
ax.set_xticks(np.arange(len(labels)), labels)
ax.yaxis.set_major_formatter(ticker.StrMethodFormatter("{x:,.0f}"))
ax.set(ylabel="Change in modeled margin (CAD thousand)",
       title=f"Conservative finish · {len(bridge)} hypothetical additional sales")
plt.show()

Use the finish-cost range to set a conditional value for further investigation. The sensitivity holds placement count and already-incurred costs fixed. The baseline veterinary and mortality ledgers contain no evidence of an intervention success rate. The case needs attainable sale weights, post-event stay and feeding records, treatment chronology, incremental pen capacity, and settlement detail before commercial application.

For future cross-notebook reporting, use the Phase 2B-3 `persistent_growth(..., landmark_ranks=True)` screen. It uses contemporaneously measured peers by lot, stage and actual weigh date, with ties retained at the empirical tenth-percentile cutoff. The screen flags 165 of 14,794 cattle with three observed intervals. Phase 2B-2 used a retrospective complete-case peer set and retains its published 162.